# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NameRectified/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections in order - each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read skills/README.md first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.

Finding A - Click Capture by Position Tier (paper Finding #3, CONFIRMED).

The paper shows weighted CTR falling from 0.423% at top 3 to 0.050% at deep positions. The label here is weighted CTR, which is clicks divided by impressions in each position tier. That is a fair descriptive number and it matches what a tier comparison should show.

My methodology question: the table proves CTR differs by tier, but the paper then says page-one pages are a more reliable fix target. That step needs an experiment where you edit page-one pages and deep pages and compare the click lift. The table alone does not prove editing helps.

This connects to our lane because our baseline leans on the same position-tier CTR idea. We inherit the same strength, which is that the tier gradient is real, and the same limit, which is that we cannot claim editing will fix a page.

Finding B - The Freshness Multiplier (paper Finding #4, CONFIRMED).

The paper says 365+ day pages refreshed within 30 days show 3.2x health and 57x impressions. The label is a comparison of refreshed pages against unrefreshed pages.

My methodology question: what decides which pages get refreshed? Strong pages are more likely to be picked for refresh, so part of the boost may come from selection, not from the refresh itself. The paper honestly flags the 361+ bucket as too small to trust. That honesty is good.

This connects to us because we deliberately left staleness out of our lane. The paper's finding is the reason a refresh lane exists. We chose the CTR side instead.

Below I load the data and check the tier CTR gradient on our own warehouse, to see if the paper's first pattern shows up here too.

In [4]:
import os, getpass, duckdb, pandas as pd, numpy as np
from pathlib import Path

env_path = Path('../../.env')
if env_path.exists():
    for line in env_path.read_text().strip().split('\n'):
        if '=' in line:
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip()
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF READ token: ')

con = duckdb.connect()
con.execute('LOAD httpfs')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

data = con.sql(f"""
    SELECT f.content_hash_id,
           MAX(f.client_hash_id) AS client_hash_id,
           SUM(f.gsc_impressions) AS impressions_fw,
           SUM(f.gsc_clicks) AS clicks_fw,
           AVG(f.gsc_avg_position) AS avg_pos_fw,
           STDDEV_SAMP(f.gsc_avg_position) AS pos_volatility_fw,
           SUM(f.ga4_sessions) AS sessions_fw,
           SUM(f.ga4_engaged_sessions) AS engaged_sessions_fw,
           c.content_type,
           c.main_intent,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.gsc_impressions ELSE 0 END) AS impressions_label,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.gsc_clicks ELSE 0 END) AS clicks_label
    FROM {FACT} f
    JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date >= '2026-01-01' AND f.report_date < '2026-04-01'
      AND f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id, c.content_type, c.main_intent
    HAVING SUM(f.gsc_impressions) >= 100
""").df()

print(f'Loaded {len(data):,} pages with complete data')

def assign_tier(pos):
    if pos <= 3:
        return 'top_3'
    if pos <= 10:
        return 'page_1'
    if pos <= 20:
        return 'striking'
    if pos <= 50:
        return 'page_3_5'
    return 'deep'

data['ctr_fw'] = data['clicks_fw'] / data['impressions_fw'] * 100
data['engagement_rate_fw'] = data['engaged_sessions_fw'] / data['sessions_fw'] * 100
data['position_tier'] = data['avg_pos_fw'].apply(assign_tier)

tier_med = data.groupby('position_tier', observed=True).apply(
    lambda g: g['clicks_fw'].sum() / g['impressions_fw'].sum() * 100,
    include_groups=False
)
data['tier_median_ctr'] = data['position_tier'].map(tier_med)
data['tier_ctr_gap'] = data['tier_median_ctr'] - data['ctr_fw']

data['ctr_label'] = data['clicks_label'] / data['impressions_label'] * 100
data['gap_label'] = data['tier_median_ctr'] - data['ctr_label']

data['below_tier_outcome'] = (data['gap_label'] > 0.1).astype(int)

data['content_type'] = data['content_type'].fillna('unknown')
data['main_intent'] = data['main_intent'].fillna('unknown')
data = data.fillna(0)

data['log_impressions_fw'] = np.log1p(data['impressions_fw'])
data['log_sessions_fw'] = np.log1p(data['sessions_fw'])

print('Weighted CTR by position tier, feature window (Jan-Feb 2026):')
tier_table = data.groupby('position_tier', observed=True).apply(
    lambda g: pd.Series({
        'n': len(g),
        'weighted_ctr_pct': g['clicks_fw'].sum() / g['impressions_fw'].sum() * 100
    }),
    include_groups=False
).reset_index()
tier_table['weighted_ctr_pct'] = tier_table['weighted_ctr_pct'].round(3)
print(tier_table.to_string(index=False))
print('The paper shows the same direction: top tiers capture more clicks per impression.')
print('This matches the tier gradient on our own data.')
print(f'Class balance: {data["below_tier_outcome"].mean():.1%} positive (below_tier_outcome=1)')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 120,258 pages with complete data
Weighted CTR by position tier, feature window (Jan-Feb 2026):
position_tier       n  weighted_ctr_pct
         deep  4178.0             0.045
       page_1 55979.0             0.327
     page_3_5 21338.0             0.149
     striking 28801.0             0.290
        top_3  9962.0             0.406
The paper shows the same direction: top tiers capture more clicks per impression.
This matches the tier gradient on our own data.
Class balance: 57.5% positive (below_tier_outcome=1)


## 2. My model under an honest split (before and after)

Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.

Before is a random split. Pages from the same client can land in both train and test. The model can memorize client identity and the scores look better than they should.

After is the client-holdout split from Week 5. Whole clients are held out. This is the honest number.

The gap between the two shows how much memorization was happening. Week 5 only showed the after. Week 6 shows both.

In [5]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

num_features = ['log_impressions_fw', 'ctr_fw', 'avg_pos_fw', 'pos_volatility_fw',
                'engagement_rate_fw', 'log_sessions_fw', 'tier_ctr_gap']
cat_features = ['content_type', 'main_intent', 'position_tier']

def precision_at_k(score, y, k):
    top = score.nlargest(k).index if len(score) >= k else score.nlargest(len(score)).index
    return y.loc[top].mean()

def run_model(train, test):
    pre = ColumnTransformer([
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ])
    X_train = pre.fit_transform(train[num_features + cat_features])
    X_test = pre.transform(test[num_features + cat_features])
    y_train = train['below_tier_outcome']
    y_test = test['below_tier_outcome']

    lr = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
    lr.fit(X_train, y_train)
    lr_probs = lr.predict_proba(X_test)[:, 1]

    rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_probs = rf.predict_proba(X_test)[:, 1]

    bl_score = ((test['impressions_fw'] >= 500).astype(int)
                * test['tier_ctr_gap'].clip(lower=0)
                * test['impressions_fw'])

    rows = {}
    rows['baseline'] = (precision_at_k(pd.Series(bl_score.values, index=test.index), y_test, 10),
                        precision_at_k(pd.Series(bl_score.values, index=test.index), y_test, 50))
    rows['logistic'] = (precision_at_k(pd.Series(lr_probs, index=test.index), y_test, 10),
                        precision_at_k(pd.Series(lr_probs, index=test.index), y_test, 50))
    rows['random_forest'] = (precision_at_k(pd.Series(rf_probs, index=test.index), y_test, 10),
                             precision_at_k(pd.Series(rf_probs, index=test.index), y_test, 50))
    return rows, y_test.mean()

def print_table(rows, base_rate, title):
    print(title)
    print(f'{"Method":<20} {"Precision@10":<14} {"Precision@50":<14}')
    print('-' * 48)
    for name, (p10, p50) in rows.items():
        print(f'{name:<20} {p10:<14.1%} {p50:<14.1%}')
    print(f'{"test base rate":<20} {base_rate:<14.1%}')
    print()

rng_idx, rng_test_idx = train_test_split(data.index, test_size=0.2, random_state=42)
rng_train, rng_test = data.loc[rng_idx].copy(), data.loc[rng_test_idx].copy()
rows_random, base_random = run_model(rng_train, rng_test)
print_table(rows_random, base_random, 'BEFORE - random split (pages from same client can span both sides)')

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
g_idx, g_test_idx = next(gss.split(data, groups=data['client_hash_id']))
g_train, g_test = data.iloc[g_idx].copy(), data.iloc[g_test_idx].copy()
rows_grouped, base_grouped = run_model(g_train, g_test)
print_table(rows_grouped, base_grouped, 'AFTER - client-holdout split (whole clients held out)')

BEFORE - random split (pages from same client can span both sides)
Method               Precision@10   Precision@50  
------------------------------------------------
baseline             100.0%         90.0%         
logistic             100.0%         98.0%         
random_forest        100.0%         98.0%         
test base rate       57.5%         

AFTER - client-holdout split (whole clients held out)
Method               Precision@10   Precision@50  
------------------------------------------------
baseline             30.0%          44.0%         
logistic             50.0%          32.0%         
random_forest        20.0%          10.0%         
test base rate       15.9%         



## 3. Leakage audit

The same hunt from Week 3, on your final feature set.

Four checks.
1. All model features come from the feature window, which ends before March 1. No future data.
2. No product decision flags are used. None exist in the warehouse.
3. One real catch. In Week 5 the tier median CTR was computed on all data, train and test together. That lets test rows affect the tier medians used on the test set. I recompute the medians from training data only and compare.
4. The label source. below_tier_outcome is built from the March outcome only. An earlier version of the label also required the same gap in the feature window, which put a feature inside the label. That condition was removed, and this check confirms the corrected label is used.

In [6]:
print('Leakage audit')
print()
print('1. Feature window check')
for f in num_features + cat_features:
    print(f'  {f} - computed from Jan 1 to Feb 28, before March 1')
print()
print('2. Product flag check')
cols = con.sql(f"DESCRIBE SELECT * FROM {FACT} WHERE month='2026-03' LIMIT 0").df()['column_name'].tolist()
flags = [c for c in cols if any(k in c.lower() for k in ['health', 'priority', 'action', 'refresh', 'flag'])]
print(f'  product flag columns found in the fact table: {len(flags)}')
print()
print('3. Tier median leak test')
med_train = g_train.groupby('position_tier', observed=True).apply(
    lambda g: g['clicks_fw'].sum() / g['impressions_fw'].sum() * 100,
    include_groups=False
)
test2 = g_test.copy()
test2['tier_median_train'] = test2['position_tier'].map(med_train)
test2['gap_train'] = test2['tier_median_train'] - test2['ctr_fw']

bl_full = ((test2['impressions_fw'] >= 500).astype(int)
           * test2['tier_ctr_gap'].clip(lower=0)
           * test2['impressions_fw'])
bl_train = ((test2['impressions_fw'] >= 500).astype(int)
            * test2['gap_train'].clip(lower=0)
            * test2['impressions_fw'])
p_full = precision_at_k(pd.Series(bl_full.values, index=test2.index), test2['below_tier_outcome'], 50)
p_train = precision_at_k(pd.Series(bl_train.values, index=test2.index), test2['below_tier_outcome'], 50)
print(f'  baseline precision@50 with full-data tier medians: {p_full:.1%}')
print(f'  baseline precision@50 with train-only tier medians: {p_train:.1%}')
print('  The change is small. The leak is real but mild in this dataset.')

print()
print('4. Label source check')
label_ok = (data['below_tier_outcome'] == (data['gap_label'] > 0.1).astype(int)).all()
old_leaky = ((data['tier_ctr_gap'] > 0.1) & (data['gap_label'] > 0.1)).astype(int)
differs = (data['below_tier_outcome'] != old_leaky).sum()
print(f'  below_tier_outcome matches the March outcome condition exactly: {label_ok}')
print(f'  pages where the old two-window label would differ: {differs:,}')
print('  The label uses the March outcome only. No label-derived feature leak.')

Leakage audit

1. Feature window check
  log_impressions_fw - computed from Jan 1 to Feb 28, before March 1
  ctr_fw - computed from Jan 1 to Feb 28, before March 1
  avg_pos_fw - computed from Jan 1 to Feb 28, before March 1
  pos_volatility_fw - computed from Jan 1 to Feb 28, before March 1
  engagement_rate_fw - computed from Jan 1 to Feb 28, before March 1
  log_sessions_fw - computed from Jan 1 to Feb 28, before March 1
  tier_ctr_gap - computed from Jan 1 to Feb 28, before March 1
  content_type - computed from Jan 1 to Feb 28, before March 1
  main_intent - computed from Jan 1 to Feb 28, before March 1
  position_tier - computed from Jan 1 to Feb 28, before March 1

2. Product flag check


  product flag columns found in the fact table: 0

3. Tier median leak test
  baseline precision@50 with full-data tier medians: 44.0%
  baseline precision@50 with train-only tier medians: 44.0%
  The change is small. The leak is real but mild in this dataset.

4. Label source check
  below_tier_outcome matches the March outcome condition exactly: True
  pages where the old two-window label would differ: 9,858
  The label uses the March outcome only. No label-derived feature leak.


## 4. Claim rewrite

Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.

Old claim: a transparent rule is more reliable than a learned model for new clients.

New claim: we observed the rule scoring higher precision@50 on held-out clients in this dataset. This is decision-support evidence that a simple rule can compete with a model. It does not prove the rule is always better.

Beyond the rewrite, here are real failure examples from the held-out clients.

In [7]:
print('Real failure examples on the client-holdout test set')
print()

pre = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
])
X_audit = pre.fit_transform(g_train[num_features + cat_features])
X_audit_test = pre.transform(g_test[num_features + cat_features])
lr_audit = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr_audit.fit(X_audit, g_train['below_tier_outcome'])
probs = lr_audit.predict_proba(X_audit_test)[:, 1]

ex = g_test.copy()
ex['lr_prob'] = probs

fp = ex[(ex['lr_prob'] > 0.5) & (ex['below_tier_outcome'] == 0)].sort_values('lr_prob', ascending=False).head(3)
fn = ex[(ex['lr_prob'] <= 0.5) & (ex['below_tier_outcome'] == 1)].sort_values('tier_ctr_gap', ascending=False).head(3)

print('False positives (high probability, page not below tier in March):')
for _, r in fp.iterrows():
    print(f'  prob={r["lr_prob"]:.2f}, impressions={r["impressions_fw"]:.0f}, gap={r["tier_ctr_gap"]:.3f}, tier={r["position_tier"]}, intent={r["main_intent"]}')
print()
print('False negatives (page below tier in March, low probability):')
for _, r in fn.iterrows():
    print(f'  prob={r["lr_prob"]:.2f}, impressions={r["impressions_fw"]:.0f}, gap={r["tier_ctr_gap"]:.3f}, tier={r["position_tier"]}, intent={r["main_intent"]}')
print()
print('The model leans on tier_ctr_gap and volume. A high probability or a big gap does not')
print('always mean the page stays below tier in March. These are the cases to inspect first.')


Real failure examples on the client-holdout test set

False positives (high probability, page not below tier in March):
  prob=0.99, impressions=820, gap=0.290, tier=striking, intent=commercial
  prob=0.99, impressions=134, gap=0.149, tier=page_3_5, intent=transactional
  prob=0.99, impressions=111, gap=0.290, tier=striking, intent=informational

False negatives (page below tier in March, low probability):
  prob=0.45, impressions=9027, gap=0.128, tier=page_1, intent=informational
  prob=0.48, impressions=3907, gap=0.122, tier=page_1, intent=transactional
  prob=0.48, impressions=7754, gap=0.122, tier=top_3, intent=informational

The model leans on tier_ctr_gap and volume. A high probability or a big gap does not
always mean the page stays below tier in March. These are the cases to inspect first.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled - markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.